download dataset from kaggle

In [1]:
import kagglehub
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

path = kagglehub.dataset_download("marcopale/housing")

In [2]:
import os
print(os.listdir(path))


['train.csv', 'test.csv', 'target.csv', 'AmesHousing.csv']


load the dataset

In [3]:
from kagglehub import KaggleDatasetAdapter



print("Path to dataset files:", path)

train_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "marcopale/housing",
  'train.csv',
)

test_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "marcopale/housing",
  'test.csv',
)



Path to dataset files: /home/tensorcore/.cache/kagglehub/datasets/marcopale/housing/versions/2


In [4]:
import numpy as np
import pandas as pd

summary = []

for col in train_df.columns:

    missing = train_df[col].isna().sum()

    if pd.api.types.is_numeric_dtype(train_df[col]):
        inf_count = np.isinf(train_df[col]).sum()
    else:
        inf_count = 0

    empty_count = (
        train_df[col].astype("string").str.strip().eq("").sum()
    )

    invalid_strings = ["?", "NA", "N/A", "null", "NULL", "None", "none"]

    invalid_count = (
        train_df[col]
        .astype("string")
        .str.strip()
        .isin(invalid_strings)
        .sum()
    )

    unique_count = train_df[col].nunique(dropna=False)

    unusable = missing + inf_count + empty_count + invalid_count

    usable = len(train_df) - unusable

    summary.append({
        "column": col,
        "dtype": train_df[col].dtype,
        "unique_values": unique_count,
        "missing": missing,
        "inf": inf_count,
        "empty": empty_count,
        "invalid_strings": invalid_count,
        "usable": usable
    })

summary_train_df = pd.DataFrame(summary)

summary_train_df


,column,dtype,unique_values,missing,inf,empty,invalid_strings,usable
0,Order,int64,2197,0,0,0,0,2197
1,PID,int64,2197,0,0,0,0,2197
2,MS SubClass,int64,15,0,0,0,0,2197
3,MS Zoning,object,7,0,0,0,0,2197
4,Lot Frontage,float64,123,362,0,0,0,1835
...,...,...,...,...,...,...,...,...
77,Mo Sold,int64,12,0,0,0,0,2197
78,Yr Sold,int64,5,0,0,0,0,2197
79,Sale Type,object,10,0,0,0,0,2197
80,Sale Condition,object,6,0,0,0,0,2197


In [5]:
numerical_cols = train_df.select_dtypes(include="number").columns

print("=== NaN ===")
print(train_df[numerical_cols].isna().sum()[
    train_df[numerical_cols].isna().sum() > 0
])

print("\n=== Infinity ===")
print(np.isinf(train_df[numerical_cols]).sum()[
    np.isinf(train_df[numerical_cols]).sum() > 0
])


print("\n=== columns with problem ===")
numerical_cols = train_df.select_dtypes(include="number").columns
columns_with_missing = []
for col in numerical_cols:
    missing_count = train_df[col].isna().sum()

    if missing_count > 0:
        columns_with_missing.append(col)

print(columns_with_missing)


=== NaN ===
Lot Frontage      362
Mas Vnr Area       22
BsmtFin SF 1        1
BsmtFin SF 2        1
Bsmt Unf SF         1
Total Bsmt SF       1
Bsmt Full Bath      1
Bsmt Half Bath      1
Garage Yr Blt     122
Garage Cars         1
Garage Area         1
dtype: int64

=== Infinity ===
Series([], dtype: int64)

=== columns with problem ===
['Lot Frontage', 'Mas Vnr Area', 'BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Garage Yr Blt', 'Garage Cars', 'Garage Area']


remove the non usable datas

In [6]:

print(f"shape before removing missing data : {train_df.shape}")
train_df = train_df.dropna(subset=columns_with_missing)
print(f"shape after removing missing data : {train_df.shape}")


shape before removing missing data : (2197, 82)
shape after removing missing data : (1705, 82)


In [7]:

categorical_cols = train_df.select_dtypes(exclude="number").columns

for col in categorical_cols:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(f"Unique values: {train_df[col].nunique(dropna=False)}")
    print(f"{'='*50}")
    
    print(train_df[col].unique())


Column: MS Zoning
Unique values: 7
['RL' 'RM' 'FV' 'RH' 'I (all)' 'C (all)' 'A (agr)']

Column: Street
Unique values: 2
['Pave' 'Grvl']

Column: Alley
Unique values: 3
[nan 'Grvl' 'Pave']

Column: Lot Shape
Unique values: 4
['Reg' 'IR1' 'IR2' 'IR3']

Column: Land Contour
Unique values: 4
['Lvl' 'HLS' 'Low' 'Bnk']

Column: Utilities
Unique values: 2
['AllPub' 'NoSewr']

Column: Lot Config
Unique values: 5
['Corner' 'CulDSac' 'Inside' 'FR2' 'FR3']

Column: Land Slope
Unique values: 3
['Gtl' 'Mod' 'Sev']

Column: Neighborhood
Unique values: 26
['SawyerW' 'NridgHt' 'Gilbert' 'NAmes' 'MeadowV' 'CollgCr' 'NWAmes'
 'Timber' 'OldTown' 'Mitchel' 'BrkSide' 'Sawyer' 'StoneBr' 'NoRidge'
 'Somerst' 'Crawfor' 'Edwards' 'SWISU' 'Blmngtn' 'IDOTRR' 'Veenker'
 'NPkVill' 'BrDale' 'ClearCr' 'Greens' 'Blueste']

Column: Condition 1
Unique values: 9
['Norm' 'RRAn' 'Feedr' 'RRNe' 'Artery' 'RRAe' 'PosN' 'PosA' 'RRNn']

Column: Condition 2
Unique values: 6
['Norm' 'RRNn' 'PosN' 'Feedr' 'PosA' 'Artery']

Colum

specify numbers for ordinal columns

In [8]:
quality_columns = [
    "Exter Qual",
    "Exter Cond",
    "Bsmt Qual",
    "Bsmt Cond",
    "Heating QC",
    "Kitchen Qual",
    "Fireplace Qu",
    "Pool QC",
    "Garage Qual",
    "Garage Cond"
]

quality_map = {
    "Ex": 5,
    "Gd": 4,
    "TA": 3,
    "Fa": 2,
    "Po": 1
}

for col in quality_columns:
    train_df[col] = train_df[col].map(quality_map).fillna(0)



bsmt_exposure_map = {
    "No": 1,
    "Mn": 2,
    "Av": 3,
    "Gd": 4
}

train_df["Bsmt Exposure"] = train_df["Bsmt Exposure"].map(bsmt_exposure_map).fillna(0)

bsmt_fin_map = {
    "Unf": 1,
    "LwQ": 2,
    "Rec": 3,
    "BLQ": 4,
    "ALQ": 5,
    "GLQ": 6
}

for col in ["BsmtFin Type 1", "BsmtFin Type 2"]:
    train_df[col] = train_df[col].map(bsmt_fin_map).fillna(0)


lot_shape_map = {
    "IR3": 1,
    "IR2": 2,
    "IR1": 3,
    "Reg": 4
}

train_df["Lot Shape"] = train_df["Lot Shape"].map(lot_shape_map).fillna(0)

land_slope_map = {
    "Gtl": 1,
    "Mod": 2,
    "Sev": 3
}

train_df["Land Slope"] = train_df["Land Slope"].map(land_slope_map).fillna(0)

functional_map = {
    "Sal": 1,
    "Maj2": 2,
    "Maj1": 3,
    "Mod": 4,
    "Min2": 5,
    "Min1": 6,
    "Typ": 7
}

train_df["Functional"] = train_df["Functional"].map(functional_map).fillna(0)

garage_finish_map = {
    "Unf": 1,
    "RFn": 2,
    "Fin": 3
}

train_df["Garage Finish"] = train_df["Garage Finish"].map(garage_finish_map).fillna(0)

paved_drive_map = {
    "N": 0,
    "P": 1,
    "Y": 2
}

train_df["Paved Drive"] = train_df["Paved Drive"].map(paved_drive_map).fillna(0)

fence_map = {
    "MnWw": 1,
    "GdWo": 2,
    "MnPrv": 3,
    "GdPrv": 4
}

train_df["Fence"] = train_df["Fence"].map(fence_map).fillna(0)

central_air_map = {
    "Y":1,
    "N":0
}

train_df["Central Air"] = train_df["Central Air"].map(central_air_map).fillna(0)


One-Hot Encoding for non ordinal and non numerical columns

In [9]:
nominal_columns = [
    "MS Zoning",
    "Street",
    "Alley",
    "Land Contour",
    "Utilities",
    "Lot Config",
    "Neighborhood",
    "Condition 1",
    "Condition 2",
    "Bldg Type",
    "House Style",
    "Roof Style",
    "Roof Matl",
    "Exterior 1st",
    "Exterior 2nd",
    "Mas Vnr Type",
    "Foundation",
    "Heating",
    "Electrical",
    "Garage Type",
    "Misc Feature",
    "Sale Type",
    "Sale Condition"
]

train_df = pd.get_dummies(
    train_df,
    columns=nominal_columns,
    dtype=int
)

ready test dataset


In [10]:
numerical_cols = test_df.select_dtypes(include="number").columns

print("=== NaN ===")
print(test_df[numerical_cols].isna().sum()[
    test_df[numerical_cols].isna().sum() > 0
])

print("\n=== Infinity ===")
print(np.isinf(test_df[numerical_cols]).sum()[
    np.isinf(test_df[numerical_cols]).sum() > 0
])


print("\n=== columns with problem ===")
numerical_cols = test_df.select_dtypes(include="number").columns
columns_with_missing = []
for col in numerical_cols:
    missing_count = test_df[col].isna().sum()

    if missing_count > 0:
        columns_with_missing.append(col)

print(columns_with_missing)

print(f"shape before removing missing data : {test_df.shape}")
test_df = test_df.dropna(subset=columns_with_missing)
print(f"shape after removing missing data : {test_df.shape}")


=== NaN ===
Lot Frontage      128
Mas Vnr Area        1
Bsmt Full Bath      1
Bsmt Half Bath      1
Garage Yr Blt      37
dtype: int64

=== Infinity ===
Series([], dtype: int64)

=== columns with problem ===
['Lot Frontage', 'Mas Vnr Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Garage Yr Blt']
shape before removing missing data : (733, 81)
shape after removing missing data : (569, 81)


In [11]:
quality_columns = [
    "Exter Qual",
    "Exter Cond",
    "Bsmt Qual",
    "Bsmt Cond",
    "Heating QC",
    "Kitchen Qual",
    "Fireplace Qu",
    "Pool QC",
    "Garage Qual",
    "Garage Cond"
]

quality_map = {
    "Ex": 5,
    "Gd": 4,
    "TA": 3,
    "Fa": 2,
    "Po": 1
}

for col in quality_columns:
    test_df[col] = test_df[col].map(quality_map).fillna(0)



bsmt_exposure_map = {
    "No": 1,
    "Mn": 2,
    "Av": 3,
    "Gd": 4
}

test_df["Bsmt Exposure"] = test_df["Bsmt Exposure"].map(bsmt_exposure_map).fillna(0)

bsmt_fin_map = {
    "Unf": 1,
    "LwQ": 2,
    "Rec": 3,
    "BLQ": 4,
    "ALQ": 5,
    "GLQ": 6
}

for col in ["BsmtFin Type 1", "BsmtFin Type 2"]:
    test_df[col] = test_df[col].map(bsmt_fin_map).fillna(0)


lot_shape_map = {
    "IR3": 1,
    "IR2": 2,
    "IR1": 3,
    "Reg": 4
}

test_df["Lot Shape"] = test_df["Lot Shape"].map(lot_shape_map).fillna(0)

land_slope_map = {
    "Gtl": 1,
    "Mod": 2,
    "Sev": 3
}

test_df["Land Slope"] = test_df["Land Slope"].map(land_slope_map).fillna(0)

functional_map = {
    "Sal": 1,
    "Maj2": 2,
    "Maj1": 3,
    "Mod": 4,
    "Min2": 5,
    "Min1": 6,
    "Typ": 7
}

test_df["Functional"] = test_df["Functional"].map(functional_map).fillna(0)

garage_finish_map = {
    "Unf": 1,
    "RFn": 2,
    "Fin": 3
}

test_df["Garage Finish"] = test_df["Garage Finish"].map(garage_finish_map).fillna(0)

paved_drive_map = {
    "N": 0,
    "P": 1,
    "Y": 2
}

test_df["Paved Drive"] = test_df["Paved Drive"].map(paved_drive_map).fillna(0)

fence_map = {
    "MnWw": 1,
    "GdWo": 2,
    "MnPrv": 3,
    "GdPrv": 4
}

test_df["Fence"] = test_df["Fence"].map(fence_map).fillna(0)

central_air_map = {
    "Y":1,
    "N":0
}

test_df["Central Air"] = test_df["Central Air"].map(central_air_map).fillna(0)

In [12]:
nominal_columns = [
    "MS Zoning",
    "Street",
    "Alley",
    "Land Contour",
    "Utilities",
    "Lot Config",
    "Neighborhood",
    "Condition 1",
    "Condition 2",
    "Bldg Type",
    "House Style",
    "Roof Style",
    "Roof Matl",
    "Exterior 1st",
    "Exterior 2nd",
    "Mas Vnr Type",
    "Foundation",
    "Heating",
    "Electrical",
    "Garage Type",
    "Misc Feature",
    "Sale Type",
    "Sale Condition"
]

test_df = pd.get_dummies(
    test_df,
    columns=nominal_columns,
    dtype=int
)

In [13]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

Train shape: (1705, 225)
Test shape: (569, 201)

Train columns:
['Order', 'PID', 'MS SubClass', 'Lot Frontage', 'Lot Area', 'Lot Shape', 'Land Slope', 'Overall Qual', 'Overall Cond', 'Year Built', 'Year Remod/Add', 'Mas Vnr Area', 'Exter Qual', 'Exter Cond', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin SF 1', 'BsmtFin Type 2', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', 'Heating QC', 'Central Air', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath', 'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'Kitchen Qual', 'TotRms AbvGrd', 'Functional', 'Fireplaces', 'Fireplace Qu', 'Garage Yr Blt', 'Garage Finish', 'Garage Cars', 'Garage Area', 'Garage Qual', 'Garage Cond', 'Paved Drive', 'Wood Deck SF', 'Open Porch SF', 'Enclosed Porch', '3Ssn Porch', 'Screen Porch', 'Pool Area', 'Pool QC', 'Fence', 'Misc Val', 'Mo Sold', 'Yr Sold', 'SalePrice', 'MS Zoning_A (agr)', 'MS Zoning_C (all)', 'MS Zoning_FV', 'MS Zo

In [ ]:
X_train = train_df.drop("SalePrice", axis=1)
y_train = train_df["SalePrice"]

X_test = test_df.copy()

X_train, X_test = X_train.align(
    X_test,
    join="outer",
    axis=1,
    fill_value=0
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("Same columns:", X_train.columns.equals(X_test.columns))
print("Non-numeric:", X_train.select_dtypes(exclude="number").columns.tolist())
print("Missing values:", X_train.isna().sum().sum())

X_train: (1705, 224)
X_test: (569, 224)
Same columns: True
2      1
3      1
4      1
5      1
6      1
      ..
728    1
729    1
730    1
731    1
732    1
Name: Central Air, Length: 569, dtype: int64
Non-numeric: []
Missing values: 0
